# 04 - Model check before training

Checks of the convolutional-recurrent classifier adapted from CRED (`wqu.models.cred`, parameters in the `model`
section of `configs/default.yaml`) on a batch of the stored features of the development subset:
1. architecture and number of parameters;
2. output shape;
3. time of one training step on the available devices (CPU, Apple GPU), and the resulting time per epoch on ESM25;
4. MC dropout: the stochastic passes must differ, the deterministic pass must not;
5. Grad-CAM: maps of both classes over MC dropout passes, over the image and the waveform (mechanics only: the network
   is not trained yet, so the maps carry no meaning).

The batch is standardized with its own statistics only for this check; the training will use per-frequency statistics
computed on the training set.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from wqu.config import load_config
from wqu.data.loading import CV_COMPONENTS
from wqu.models.cred import build_model
from wqu.data.waveforms import load_waveforms, stored_sampling_rate_hz
from wqu.models.diagnostics import available_devices, count_parameters, time_training_step
from wqu.models.explain import grad_cam_by_class
from wqu.models.loss import bad_class_weight, weighted_bce_loss
from wqu.models.uncertainty import mc_dropout_probabilities
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    REPRESENTATIONS,
    feature_file_path,
    grid_from_config,
    process_record,
    read_features,
)
from wqu.visualization import apply_plot_style, plot_grad_cam, save_figure

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FEATURE_SOURCE = "dev_subset"
BATCH_SIZE = 64
N_TIMED_STEPS = 5
N_ESM25_RECORDS = 100_249

config = load_config(CONFIG_PATH)
columns = config.columns
preprocessing = config.preprocessing
model_config = config.model
model_config.tables_dir.mkdir(parents=True, exist_ok=True)
model_config.figures_dir.mkdir(parents=True, exist_ok=True)
apply_plot_style()
torch.manual_seed(config.split.random_seed)

features_path = feature_file_path(preprocessing, FEATURE_SOURCE)
stored_records = pd.read_csv(features_path.with_name("records.csv"), index_col=0)
print(f"Feature file: {features_path} ({len(stored_records):,} records)")

## 1. Batch of stored features

In [ ]:
batch_records = stored_records.groupby(columns.label, group_keys=False).sample(
    n=BATCH_SIZE // 2, random_state=config.split.random_seed
)
batch_labels = torch.tensor(batch_records[columns.label].to_numpy(), dtype=torch.float32)
batches = {}
for representation in REPRESENTATIONS:
    values, grid = read_features(features_path, representation, batch_records[FEATURE_INDEX_COLUMN].to_numpy())
    batches[representation] = torch.from_numpy((values - values.mean()) / values.std())
print({name: tuple(batch.shape) for name, batch in batches.items()})
print(f"Bad-class weight on the whole development subset (for this check only): "
      f"{bad_class_weight(stored_records[columns.label].to_numpy()):.2f}")

## 2. Architecture, parameters and output shape

In [ ]:
model = build_model(model_config, len(CV_COMPONENTS), len(grid.frequencies_hz))
print(model)
print(f"Trainable parameters: {count_parameters(model):,}")
with torch.no_grad():
    logits = model.eval()(batches["spectrogram"])
print(f"Output shape: {tuple(logits.shape)} (one logit per record)")

## 3. Training speed

Time of one training step (forward, backward, update) on a batch of `BATCH_SIZE` records, and the time of one epoch
over the training part of ESM25 (`1 - split.validation_fraction` of all records).

In [ ]:
loss = weighted_bce_loss(bad_class_weight(stored_records[columns.label].to_numpy()))
n_training_records = int(N_ESM25_RECORDS * (1 - config.split.validation_fraction))
rows = []
for device in available_devices():
    seconds = time_training_step(
        build_model(model_config, len(CV_COMPONENTS), len(grid.frequencies_hz)),
        loss, batches["spectrogram"], batch_labels, device, N_TIMED_STEPS,
    )
    rows.append({"device": device, "seconds_per_step": seconds,
                 "minutes_per_epoch": seconds * np.ceil(n_training_records / BATCH_SIZE) / 60.0})
timing = pd.DataFrame(rows).set_index("device")
timing.to_csv(model_config.tables_dir / "training_speed.csv")
timing.round(3)

## 4. MC dropout

With dropout active the passes on the same input must differ (non-zero spread); with the model in evaluation mode they
must be identical. The spread of an untrained network says nothing about uncertainty: this only checks the mechanics.

In [ ]:
model = model.to("cpu")
stochastic = mc_dropout_probabilities(model, batches["spectrogram"], model_config.mc_samples)
with torch.no_grad():
    model.eval()
    deterministic = torch.stack([torch.sigmoid(model(batches["spectrogram"])) for _ in range(3)])
print(f"MC dropout: {model_config.mc_samples} passes, mean spread across passes "
      f"{stochastic.std(dim=0).mean().item():.4f}")
print(f"Evaluation mode: largest difference between passes {(deterministic - deterministic[0]).abs().max().item():.2e}")

## 5. Grad-CAM

Maps of the bad class (logit) and of the good class (opposite of the logit) on the last convolutional layer, over
`model.grad_cam.mc_samples` MC dropout passes, for `model.grad_cam.records_per_class` records per class. Top: input image
with the contours of the mean maps; middle: mean and standard deviation over the passes; bottom: waveform of the window
with the mean of each map over frequencies. With an untrained network this only checks that the maps are computed and
aligned with the image and the waveform.

In [ ]:
explained_records = batch_records.groupby(columns.label, group_keys=False).head(model_config.grad_cam_records_per_class)
positions = [batch_records.index.get_loc(index) for index in explained_records.index]
explained_inputs = batches["spectrogram"][positions]
maps = grad_cam_by_class(model, model.last_conv_layer, explained_inputs, model_config.grad_cam_mc_samples)

raw_images, _ = read_features(features_path, "spectrogram", explained_records[FEATURE_INDEX_COLUMN].to_numpy())
traces = load_waveforms(explained_records, config)
sampling_rate_hz = stored_sampling_rate_hz(stored_records)
preprocessing_grid = grid_from_config(preprocessing)
for row, ((index, record), trace) in enumerate(zip(explained_records.iterrows(), traces)):
    window_waveform = process_record(
        trace, record, sampling_rate_hz, preprocessing, preprocessing_grid, config.waveforms.zero_tolerance
    ).window_waveform
    figure = plot_grad_cam(
        raw_images[row], grid.frequencies_hz, grid.frequency_edges_hz, grid.times_s,
        {name: (result.mean[row].numpy(), result.std[row].numpy()) for name, result in maps.items()},
        window_waveform, preprocessing.target_sampling_rate_hz, CV_COMPONENTS,
        f"Grad-CAM (untrained network), label {record[columns.label]}: {record['trace_name_old']}",
    )
    save_figure(figure, model_config.figures_dir / f"grad_cam_check_{record['trace_name_old']}.png")